In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!unzip '/content/drive/MyDrive/archive.zip'

Streaming output truncated to the last 5000 lines.
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_1042_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_1044_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_1046_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_1047_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_1049_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_104_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_1050_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_1063_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_1068_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_1071_notdrowsy.jpg  
  inflating: NTHU-DDD/notdrowsy/002_noglasses_nonsleepyCombination_107

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = 224
batch_size = 32

data_dir = "/content/NTHU-DDD"   # folder that contains drowsy/ and non-drowsy/

# Create generator with validation split
datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    validation_split=0.2   # 80% train, 20% test
)

# TRAIN generator
train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',   # important
    shuffle=True
)

# TEST generator
test_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',  # important
    shuffle=False
)

Found 14400 images belonging to 2 classes.
Found 3600 images belonging to 2 classes.


In [ ]:
num_classes = train_generator.num_classes
print("Number of classes:", num_classes)
print("Class labels:", train_generator.class_indices)


Number of classes: 2
Class labels: {'drowsy': 0, 'notdrowsy': 1}


**CNN**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),

    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    44,302,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │         1,026 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,398,018 (169.36 MB)

 Trainable params: 44,397,570 (169.36 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
cnn_history = cnn_model.fit(
    train_generator,
    epochs=30
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 201s 424ms/step - accuracy: 0.5962 - loss: 5.1930
Epoch 2/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 186s 413ms/step - accuracy: 0.6924 - loss: 0.5601
Epoch 3/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 189s 420ms/step - accuracy: 0.7450 - loss: 0.5065
Epoch 4/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 189s 419ms/step - accuracy: 0.7753 - loss: 0.4716
Epoch 5/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 187s 416ms/step - accuracy: 0.7904 - loss: 0.4244
Epoch 6/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 192s 427ms/step - accuracy: 0.7880 - loss: 0.4243
Epoch 7/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 195s 433ms/step - accuracy: 0.7943 - loss: 0.4266
Epoch 8/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 189s 420ms/step - accuracy: 0.7973 - loss: 0.4125
Epoch 9/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 187s 414ms/step - accuracy: 0.7877 - loss: 0.4429
Epoch 10/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 187s 417ms/step - accuracy: 0.8187 - loss: 0.3785
Epoch 11/30
450/450 ━━━━━━━━━━━━━━━━━━━━ 194s 432ms/step - accuracy: 0.8211 - loss: 0.3548
Epoch 12

**DENSENET121**

In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam


In [ ]:
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

for layer in base_model.layers:
    layer.trainable = False


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(train_generator.num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 7,300,418 (27.85 MB)

 Trainable params: 262,914 (1.00 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=30,
    steps_per_epoch=70
)

Epoch 1/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.8158 - loss: 0.3811 - val_accuracy: 0.5917 - val_loss: 0.7840
Epoch 2/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.8470 - loss: 0.3513 - val_accuracy: 0.5842 - val_loss: 0.7938
Epoch 3/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.8284 - loss: 0.3628 - val_accuracy: 0.5925 - val_loss: 0.8278
Epoch 4/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.8426 - loss: 0.3590 - val_accuracy: 0.6092 - val_loss: 0.7740
Epoch 5/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 74s 1s/step - accuracy: 0.8325 - loss: 0.3698 - val_accuracy: 0.6053 - val_loss: 0.8014
Epoch 6/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.8432 - loss: 0.3505 - val_accuracy: 0.6058 - val_loss: 0.7548
Epoch 7/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 57s 818ms/step - accuracy: 0.8461 - loss: 0.3559 - val_accuracy: 0.6000 - val_loss: 0.7840
Epoch 8/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.8366 - loss: 0.3614 - val_accuracy: 0.6050 - val_lo

**MOBILENET**

In [ ]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model

# Load MobileNet base
base_mobilenet = MobileNet(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base layers
for layer in base_mobilenet.layers:
    layer.trainable = False

# Custom head
x = Flatten()(base_mobilenet.output)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

# Model
mobilenet_model = Model(
    inputs=base_mobilenet.input,
    outputs=output
)

# Compile
mobilenet_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Summary
mobilenet_model.summary()


17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 112, 112, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_bn (BatchNormalization)   │ (None, 112, 112, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_relu (ReLU)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1 (DepthwiseConv2D)     │ (None, 112, 112, 32)   │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_bn                    │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_relu (ReLU)           │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1 (Conv2D)              │ (None, 112, 112, 64)   │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_bn                    │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_relu (ReLU)           │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pad_2 (ZeroPadding2D)      │ (None, 113, 113, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2 (DepthwiseConv2D)     │ (None, 56, 56, 64)     │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_bn                    │ (None, 56, 56, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_relu (ReLU)           │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2 (Conv2D)              │ (None, 56, 56, 128)    │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_bn                    │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_relu (ReLU)           │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3 (DepthwiseConv2D)     │ (None, 56, 56, 128)    │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_bn                    │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_relu (ReLU)           │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3 (Conv2D)              │ (None, 56, 56, 128)    │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3_bn                    │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 16,074,690 (61.32 MB)

 Trainable params: 12,845,826 (49.00 MB)

 Non-trainable params: 3,228,864 (12.32 MB)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=30,
    steps_per_epoch=70
)

Epoch 1/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.8478 - loss: 0.3260 - val_accuracy: 0.6197 - val_loss: 0.8721
Epoch 2/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.8562 - loss: 0.3191 - val_accuracy: 0.6208 - val_loss: 0.9650
Epoch 3/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.8620 - loss: 0.3048 - val_accuracy: 0.6167 - val_loss: 0.9646
Epoch 4/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.8689 - loss: 0.2989 - val_accuracy: 0.6083 - val_loss: 0.9970
Epoch 5/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.8477 - loss: 0.3416 - val_accuracy: 0.6203 - val_loss: 0.9115
Epoch 6/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.8526 - loss: 0.3150 - val_accuracy: 0.6253 - val_loss: 0.8683
Epoch 7/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 53s 766ms/step - accuracy: 0.8660 - loss: 0.2982 - val_accuracy: 0.6231 - val_loss: 0.8768
Epoch 8/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.8702 - loss: 0.2888 - val_accuracy: 0.6108 - val_lo

**LENET**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# LeNet model
lenet_model = Sequential()

lenet_model.add(Conv2D(6, (5,5), activation="relu", input_shape=(224,224,3)))
lenet_model.add(MaxPooling2D(pool_size=(2,2)))
lenet_model.add(Conv2D(16, (5,5), activation="relu"))
lenet_model.add(MaxPooling2D(pool_size=(2,2)))
lenet_model.add(Flatten())
lenet_model.add(Dense(120, activation="relu"))
lenet_model.add(Dense(84, activation="relu"))
lenet_model.add(Dense(num_classes, activation="softmax"))

lenet_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
lenet_model.summary()



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 220, 220, 6)    │           456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 110, 110, 6)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 106, 106, 16)   │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 53, 53, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 44944)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 120)            │     5,393,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 2)              │           170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,406,606 (20.62 MB)

 Trainable params: 5,406,606 (20.62 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=30,
    steps_per_epoch=70
)

Epoch 1/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.8611 - loss: 0.3112 - val_accuracy: 0.6247 - val_loss: 0.9803
Epoch 2/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.8790 - loss: 0.2824 - val_accuracy: 0.5978 - val_loss: 1.1055
Epoch 3/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.8834 - loss: 0.2613 - val_accuracy: 0.6300 - val_loss: 1.0271
Epoch 4/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.8684 - loss: 0.2784 - val_accuracy: 0.6283 - val_loss: 1.0409
Epoch 5/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.8652 - loss: 0.2879 - val_accuracy: 0.6036 - val_loss: 1.1768
Epoch 6/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.8800 - loss: 0.2741 - val_accuracy: 0.6119 - val_loss: 1.1004
Epoch 7/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 56s 808ms/step - accuracy: 0.8762 - loss: 0.2922 - val_accuracy: 0.6156 - val_loss: 1.0553
Epoch 8/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.8874 - loss: 0.2503 - val_accuracy: 0.6122 - val_lo

**RNN**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Reshape, SimpleRNN, Dense, Dropout

img_size = 224
num_classes = train_generator.num_classes  # auto-detect classes

rnn_model = Sequential([

    # Convert image to sequence
    Reshape((224, 224*3), input_shape=(224,224,3)),

    # RNN Layers
    SimpleRNN(128, activation='tanh', return_sequences=True),
    Dropout(0.5),

    SimpleRNN(64, activation='tanh'),
    Dropout(0.5),

    # Output Layer
    Dense(num_classes, activation='softmax')
])

rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

rnn_model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_3 (Reshape)             │ (None, 224, 672)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 224, 128)       │       102,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 224, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 115,010 (449.26 KB)

 Trainable params: 115,010 (449.26 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Training
rnn_history = rnn_model.fit(
    train_generator,
    epochs=30,
    steps_per_epoch=70
)

Epoch 1/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 6s 81ms/step - accuracy: 0.6897 - loss: 0.5993
Epoch 2/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 7s 96ms/step - accuracy: 0.7259 - loss: 0.5517
Epoch 3/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 5s 79ms/step - accuracy: 0.7800 - loss: 0.4756
Epoch 4/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 7s 99ms/step - accuracy: 0.7280 - loss: 0.5406
Epoch 5/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 5s 78ms/step - accuracy: 0.7848 - loss: 0.4629
Epoch 6/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 7s 98ms/step - accuracy: 0.7712 - loss: 0.5073
Epoch 7/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8134 - loss: 0.4264
Epoch 8/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 7s 82ms/step - accuracy: 0.7235 - loss: 0.5636
Epoch 9/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 7s 101ms/step - accuracy: 0.8137 - loss: 0.4294
Epoch 10/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 6s 79ms/step - accuracy: 0.7835 - loss: 0.4697
Epoch 11/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 7s 101ms/step - accuracy: 0.8300 - loss: 0.4109
Epoch 12/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 6s 82ms/step - accurac

**Densenet121+Mobilenet**

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121, MobileNet
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.layers import Dropout, Concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

img_size = 224
num_classes = train_generator.num_classes

# ===============================
# 1️⃣ Create Input Layer
# ===============================
input_layer = Input(shape=(224,224,3))

# ===============================
# 2️⃣ Load Models WITHOUT input_tensor
# ===============================
densenet_base = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

mobilenet_base = MobileNet(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze layers
for layer in densenet_base.layers:
    layer.trainable = False

for layer in mobilenet_base.layers:
    layer.trainable = False

# ===============================
# 3️⃣ Pass SAME input through both models
# ===============================
x1 = densenet_base(input_layer)
x1 = GlobalAveragePooling2D()(x1)

x2 = mobilenet_base(input_layer)
x2 = GlobalAveragePooling2D()(x2)

# ===============================
# 4️⃣ Feature Fusion
# ===============================
combined = Concatenate()([x1, x2])

# ===============================
# 5️⃣ Classification Head
# ===============================
x = Dense(512, activation="relu")(combined)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

output = Dense(num_classes, activation="softmax")(x)

# ===============================
# 6️⃣ Final Model
# ===============================
hybrid_model = Model(inputs=input_layer, outputs=output)

hybrid_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

hybrid_model.summary()

Model: "functional_47"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_15      │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ densenet121         │ (None, 7, 7,      │  7,037,504 │ input_layer_15[0… │
│ (Functional)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenet_1.00_224  │ (None, 7, 7,      │  3,228,864 │ input_layer_15[0… │
│ (Functional)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1024)      │          0 │ densenet121[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1024)      │          0 │ mobilenet_1.00_2… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 2048)      │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_31 (Dense)    │ (None, 512)       │  1,049,088 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 512)       │          0 │ dense_31[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_32 (Dense)    │ (None, 256)       │    131,328 │ dropout_19[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_20          │ (None, 256)       │          0 │ dense_32[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_33 (Dense)    │ (None, 2)         │        514 │ dropout_20[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,447,298 (43.67 MB)

 Trainable params: 1,180,930 (4.50 MB)

 Non-trainable params: 10,266,368 (39.16 MB)

In [ ]:
history = hybrid_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    steps_per_epoch=70
)

Epoch 1/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 89s 763ms/step - accuracy: 0.5322 - loss: 0.9173 - val_accuracy: 0.4975 - val_loss: 0.7362
Epoch 2/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 20s 287ms/step - accuracy: 0.6499 - loss: 0.6442 - val_accuracy: 0.5447 - val_loss: 0.6776
Epoch 3/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 19s 281ms/step - accuracy: 0.7175 - loss: 0.5389 - val_accuracy: 0.6083 - val_loss: 0.6294
Epoch 4/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 28s 407ms/step - accuracy: 0.7719 - loss: 0.4691 - val_accuracy: 0.6358 - val_loss: 0.6155
Epoch 5/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 20s 282ms/step - accuracy: 0.8176 - loss: 0.4140 - val_accuracy: 0.6739 - val_loss: 0.5810
Epoch 6/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 20s 284ms/step - accuracy: 0.8282 - loss: 0.3726 - val_accuracy: 0.6356 - val_loss: 0.6810
Epoch 7/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 15s 219ms/step - accuracy: 0.8264 - loss: 0.3860 - val_accuracy: 0.6764 - val_loss: 0.6078
Epoch 8/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 20s 283ms/step - accuracy: 0.8480 - loss: 0.3400 - val_accu

**CNN + DenseNet121 Model**

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.layers import Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling2D, Concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

img_size = 224
num_classes = train_generator.num_classes

# ===============================
# 1️⃣ Input Layer
# ===============================
input_layer = Input(shape=(224,224,3))

# ===============================
# 2️⃣ Custom CNN Branch
# ===============================
x1 = Conv2D(32, (3,3), activation='relu')(input_layer)
x1 = BatchNormalization()(x1)
x1 = MaxPooling2D()(x1)

x1 = Conv2D(64, (3,3), activation='relu')(x1)
x1 = BatchNormalization()(x1)
x1 = MaxPooling2D()(x1)

x1 = Conv2D(128, (3,3), activation='relu')(x1)
x1 = BatchNormalization()(x1)
x1 = MaxPooling2D()(x1)

x1 = Flatten()(x1)
x1 = Dense(256, activation='relu')(x1)
x1 = Dropout(0.5)(x1)

# ===============================
# 3️⃣ DenseNet121 Branch
# ===============================
base_densenet = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_tensor=input_layer
)

# Freeze DenseNet layers
for layer in base_densenet.layers:
    layer.trainable = False

x2 = base_densenet.output
x2 = GlobalAveragePooling2D()(x2)

# ===============================
# 4️⃣ Feature Fusion
# ===============================
combined = Concatenate()([x1, x2])

# ===============================
# 5️⃣ Final Classification Head
# ===============================
x = Dense(512, activation='relu')(combined)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation='softmax')(x)

# ===============================
# 6️⃣ Final Hybrid Model
# ===============================
hybrid_model = Model(inputs=input_layer, outputs=output)

hybrid_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

hybrid_model.summary()

Model: "functional_22"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_2    │ (None, 230, 230,  │          0 │ input_layer_8[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d_2… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_3    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_3… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 29,939,970 (114.21 MB)

 Trainable params: 22,902,018 (87.36 MB)

 Non-trainable params: 7,037,952 (26.85 MB)

In [ ]:
history = hybrid_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    steps_per_epoch=70
)

Epoch 1/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 73s 642ms/step - accuracy: 0.5706 - loss: 1.8876 - val_accuracy: 0.5936 - val_loss: 0.6687
Epoch 2/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 21s 304ms/step - accuracy: 0.7453 - loss: 0.5538 - val_accuracy: 0.5350 - val_loss: 0.6744
Epoch 3/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 21s 305ms/step - accuracy: 0.7950 - loss: 0.4244 - val_accuracy: 0.5125 - val_loss: 0.8379
Epoch 4/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 21s 302ms/step - accuracy: 0.8308 - loss: 0.3839 - val_accuracy: 0.5153 - val_loss: 0.8441
Epoch 5/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 21s 303ms/step - accuracy: 0.8440 - loss: 0.3321 - val_accuracy: 0.5381 - val_loss: 0.6671
Epoch 6/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 21s 304ms/step - accuracy: 0.8590 - loss: 0.3130 - val_accuracy: 0.5247 - val_loss: 0.7100
Epoch 7/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 15s 212ms/step - accuracy: 0.8486 - loss: 0.3181 - val_accuracy: 0.5197 - val_loss: 0.7030
Epoch 8/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 43s 304ms/step - accuracy: 0.8633 - loss: 0.3005 - val_accu

**LSTM**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Reshape

img_size = 224
num_classes = train_generator.num_classes   # automatically detects classes

# LSTM Model
lstm_model = Sequential()

# Reshape image to sequence format
lstm_model.add(Reshape((224, 224*3), input_shape=(224, 224, 3)))

# LSTM Layers
lstm_model.add(LSTM(128, return_sequences=True))
lstm_model.add(Dropout(0.5))

lstm_model.add(LSTM(64))
lstm_model.add(Dropout(0.5))

# Output Layer
lstm_model.add(Dense(num_classes, activation='softmax'))

# Compile
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_4 (Reshape)             │ (None, 224, 672)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 224, 128)       │       410,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 224, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 459,650 (1.75 MB)

 Trainable params: 459,650 (1.75 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = lstm_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    steps_per_epoch=70
)

Epoch 1/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 21s 230ms/step - accuracy: 0.5009 - loss: 0.7296 - val_accuracy: 0.6203 - val_loss: 0.6588
Epoch 2/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 16s 225ms/step - accuracy: 0.5654 - loss: 0.6754 - val_accuracy: 0.6836 - val_loss: 0.6008
Epoch 3/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 16s 225ms/step - accuracy: 0.6099 - loss: 0.6357 - val_accuracy: 0.6706 - val_loss: 0.6437
Epoch 4/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 17s 243ms/step - accuracy: 0.7357 - loss: 0.5225 - val_accuracy: 0.5797 - val_loss: 0.9211
Epoch 5/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 15s 217ms/step - accuracy: 0.7461 - loss: 0.5134 - val_accuracy: 0.5983 - val_loss: 0.9000
Epoch 6/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 15s 212ms/step - accuracy: 0.7736 - loss: 0.4880 - val_accuracy: 0.6956 - val_loss: 0.7736
Epoch 7/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 11s 156ms/step - accuracy: 0.7831 - loss: 0.4379 - val_accuracy: 0.6789 - val_loss: 0.8090
Epoch 8/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 15s 217ms/step - accuracy: 0.8180 - loss: 0.4139 - val_accu

**CNN+LSTM**

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.layers import TimeDistributed, Flatten
from tensorflow.keras.layers import LSTM, Dense, Dropout, Reshape
from tensorflow.keras.models import Sequential

img_size = 224
num_classes = train_generator.num_classes

# Hybrid CNN + LSTM Model
model = Sequential()

# ======================
# CNN Feature Extraction
# ======================
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(128, (3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

# ======================
# Convert CNN output to sequence
# ======================
model.add(Reshape((-1, 128)))   # Convert feature map to sequence

# ======================
# LSTM Layer
# ======================
model.add(LSTM(128))
model.add(Dropout(0.5))

# ======================
# Output Layer
# ======================
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_5 (Reshape)             │ (None, 676, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 259,266 (1012.76 KB)

 Trainable params: 258,818 (1011.01 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    steps_per_epoch=70
)

Epoch 1/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 27s 317ms/step - accuracy: 0.5362 - loss: 0.7195 - val_accuracy: 0.5000 - val_loss: 0.6918
Epoch 2/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 20s 283ms/step - accuracy: 0.6440 - loss: 0.6136 - val_accuracy: 0.5000 - val_loss: 0.7313
Epoch 3/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 20s 292ms/step - accuracy: 0.7398 - loss: 0.4966 - val_accuracy: 0.5000 - val_loss: 1.6156
Epoch 4/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 19s 276ms/step - accuracy: 0.7799 - loss: 0.4453 - val_accuracy: 0.5000 - val_loss: 1.6913
Epoch 5/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 21s 296ms/step - accuracy: 0.7993 - loss: 0.4216 - val_accuracy: 0.5000 - val_loss: 3.0536
Epoch 6/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 19s 276ms/step - accuracy: 0.8307 - loss: 0.3714 - val_accuracy: 0.5000 - val_loss: 2.2551
Epoch 7/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 14s 201ms/step - accuracy: 0.8613 - loss: 0.3155 - val_accuracy: 0.5000 - val_loss: 2.2698
Epoch 8/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 20s 290ms/step - accuracy: 0.8709 - loss: 0.3145 - val_accu

In [ ]:
pip install facenet-pytorch torch torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
import os
import torch
from facenet_pytorch import MTCNN
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mtcnn = MTCNN(device=device)

folder = "/content/NTHU-DDD"   # main dataset folder

correct = 0
total = 0

for class_name in os.listdir(folder):

    class_path = os.path.join(folder, class_name)

    if not os.path.isdir(class_path):
        continue

    for img_name in os.listdir(class_path):

        img_path = os.path.join(class_path, img_name)

        # Skip if not image file
        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        image = Image.open(img_path).convert("RGB")

        boxes, _ = mtcnn.detect(image)

        # For detection accuracy:
        # If dataset contains faces in all images
        if boxes is not None:
            correct += 1

        total += 1

accuracy = correct / total
print("MTCNN Detection Accuracy:", accuracy)

MTCNN Detection Accuracy: 0.9922777777777778


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# -----------------------------
# 1️⃣ Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 2️⃣ Hyperparameters
# -----------------------------
num_classes = 2
num_epochs = 30
batch_size = 16
learning_rate = 0.001

# -----------------------------
# 3️⃣ Transforms
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor()
])

# -----------------------------
# 4️⃣ Load Dataset
# -----------------------------
dataset = datasets.ImageFolder(
    root="/content/NTHU-DDD",   # change if needed
    transform=transform
)

# Split into Train (80%) & Validation (20%)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("Classes:", dataset.classes)
print("Training samples:", train_size)
print("Validation samples:", val_size)

# -----------------------------
# 5️⃣ Model
# -----------------------------
model = models.inception_v3(pretrained=True, aux_logits=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# -----------------------------
# 6️⃣ Loss & Optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# -----------------------------
# 7️⃣ Training
# -----------------------------
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")

print("Training Completed")

# -----------------------------
# 8️⃣ Overall Validation Accuracy
# -----------------------------
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

overall_accuracy = 100 * correct / total
print("Overall Validation Accuracy: {:.2f}%".format(overall_accuracy))